# 从 compressed IR 到二维结构：test10.mps 探索指南

本示例用 MIPFrame 的通用查询逐步验证模型结构。我们先观察 family，再检查真实行、
变量关联和精确系数，最后评估一个代入想法的非零项代价。它不运行求解器，也不修改模型。

运行前请在仓库环境中安装 mipframe，并确保 data/test10.mps 可用（约 28 MB）。
Notebook 从仓库根目录或 examples 目录启动均可。首次读取会计算 compressed IR，
需要一些时间。文中的 F0、V2 等标签只对这个模型和默认 template 配置有效。

In [ ]:
from pathlib import Path
import mipframe as mf
from mipframe import col

source = next(
    (root / "data" / "test10.mps"
     for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data" / "test10.mps").is_file()),
    None,
)
if source is None:
    raise FileNotFoundError("找不到 data/test10.mps；请从仓库目录启动 Notebook，或手动设置 source")

m = mf.read(source)
m

## 1. 先读模型地图

默认 compressed IR 展示模型规模、变量角色、约束 family 和逐层细化结果。
这里约有 19 万个变量、2303 条约束；F0 与 F1 各有 950 行，F2 有
200 行，V2 有 190000 个变量。IR 的结构是规范化多重集合，不能代替原始公式。

In [ ]:
m.ir

## 2. 把 family ID 锚定到原始成员

先看行名、变量名和非零项数。Frame 的 head() 只显示成员片段；一个小 family
在默认显示上限内会完整列出。索引显示的是原始模型 ID。

按原始 ID 查成员用 `.loc[id]`，按名称用 `.by_name(name)`；
`.iloc[位置]` 和 `.iloc[起点:终点]` 始终按当前选区的位置访问。
`m.family("F0").rows` 按族标签选择成员。已有单行对象时直接用 `row.vars`。

In [ ]:
F0 = m.family("F0@2").rows
F1 = m.family("F1@2").rows
F2 = m.family("F2@2").rows
V1 = m.family("V1@1").vars
V2 = m.family("V2@1").vars

display(F0.head(), F1.head(), F2.head())
display(V1.head(), V2.head())

## 3. 打开代表行，并检查 family 中被签名隐藏的数值变化

`m.family("F0")` 返回单行 Frame，`.rows` 是原始成员。两个默认族目录均为 depth 2；
本例解释行模板所用的变量角色显式通过 `m.families(kind="var", depth=1)` 取得。

公式始终限项展示；省略号不表示剩余系数相同。原始 F0 行的 V2 系数是变化的，
而 F1 示例是单位系数。F0 的 parameters 关联表 可给出整个 family 的精确系数分布：
规范方向的 +V2 系数有 190000 个不同值。这与 family 成员共享结构角色并不矛盾。

In [ ]:
for label in ("F0", "F1", "F2"):
    family = m.family(label, depth=2)
    row = family.rows.iloc[0]
    print(family.one().id, row.name)
    print(row.formula(order="index", max_terms=8))

F0_family = m.family("F0", depth=2)
display(F0_family.inspect(samples=0))
assert F0_family.related("parameters").where(basis="canonical", orientation=0, role="+V2", quantity="coefficient").one().varying


## 4. F0 与 F1 是否共享同一批 V2？

抽取 10 条 F0 行，通过限制在 V2 的变量关联找 F1 候选。若一条 F1
包含全部 200 个 V2，比较双方 support：预期共同 200 个变量，仅 F0
多一个 V0。这里检验的是 support 对齐，不是系数相等。

In [ ]:
for row in F0.sample(10, random_state=42):
    a = row.vars
    a_v2 = a & V2
    counts = m.matrix.nnz(rows=F1, vars=a_v2, axis="row")
    matches = counts.where(col("nnz") == len(a_v2)).refs("row")
    if len(matches) == 1:
        b = matches.one().vars
        print(row.name, "→", matches.one().name,
              "V2:", len(a_v2),
              "共同/仅F0/仅F1:", len(a & b), len(a - b), len(b - a))
    else:
        print(row.name, "→", len(matches), "个候选", "V2:", len(a_v2))


## 5. 给一个 V2 变量恢复两个坐标

变量的 rows 视图给出相邻原始行、family 和精确系数。若它恰好落在一条 F0
和一条 F2，可暂记为 x_(i,j)。先看一个变量，再随机检查 100 个。

In [ ]:
x = V2.iloc[123]
f0_edge = x.terms.where(row=F0).one()
f2_edge = x.terms.where(row=F2).one()
display(x.rows)
print({"variable": x.name, "i_row": f0_edge.row.name, "j_row": f2_edge.row.name})


In [ ]:
sample = V2.sample(100, random_state=42)
P_sample = sample.project(i=F0, j=F2)
print(f"checked={len(P_sample)}, bad=0")
display(sample.pattern_counts())


## 6. 全量核验 950 × 200 个坐标

抽样不能证明覆盖完整。下面使用公开的批量投影，检查每个 V2 是否各有一个
F0/F1/F2 坐标，再检查 F0×F2 的唯一性与完整覆盖。严格投影发现零匹配或多匹配时
抛出 CardinalityError；通过 exception.report.violations 可以查询全部违规成员和匹配边。

计算仅使用稀疏索引；不读取名称、系数，不建立完整笛卡尔积。函数依赖报告另外核验
F0 与 F1 坐标的对应关系，并注明未覆盖目标数。


In [ ]:
P = V2.project(i=F0, k=F1, j=F2, memory_mb=128)
report = P.check("product", ("i", "j"))
display(report)
report.require()
assert len(P) == P.nunique(("i", "j")) == len(F0) * len(F2) == 190000

forward = P.check("function", key="i", value="k")
backward = P.check("function", key="k", value="i")
forward.require()
backward.require()
assert all(n == 0 for n in forward.summary["uncovered_targets"].values())
assert all(n == 0 for n in backward.summary["uncovered_targets"].values())
display(forward, backward)

point = P.loc[x.id]
print(point.var.name, "↔", (point.i.id, point.j.id),
      "↔", (point.i.name, point.j.name))


In [ ]:
# 精确系数通过原始变量 Ref 关联，检查基数与完整匹配。
A = m.terms(rows=F0, vars=V2).select("var", a=col("coef"))
Q = P.join(A, on="var", cardinality="1:1", require="left")
S = Q.group_by("i").agg(distinct=col("a").nunique(), amin=col("a").min(), amax=col("a").max())
assert len(Q) == 190000 and len(S) == 950
assert S.column("distinct").max() > 1
display(S.head())


## 让 structural signature 自动提出 indexed 候选

前面的投影验证了我们手动选择的 F0/F1/F2。现在提取 read() 时已完成的 indexed 结果：各层 family 和粗分组提出候选，再按原始成员验证。颜色并不直接代表坐标轴；结果可能包含多个有效坐标视图。


In [ ]:
I = m.indexed(memory_mb=128)
display(I)
# 通过公开原始 ID 映射选择覆盖 V2、使用 F0/F2 轴的自动发现结果。
def same_ids(left, right):
    return len(left) == len(right) and not len(left - right)

T = next(t for t in (I.tensor(record.id) for record in I.tensors)
         if same_ids(t.vars, V2)
         and same_ids(t.related("axes").where(axis=0).rows, F0)
         and same_ids(t.related("axes").where(axis=1).rows, F2))
display(T)
auto_coordinates = T.related("coordinates")
product = auto_coordinates.check("product", ("i", "j"))
product.require()
display(product)
assert len(auto_coordinates) == auto_coordinates.nunique(("i", "j")) == 190000
assert list(T.related("dimensions").where(component="shape").column("size")) == [950, 200] and T.one().complete_product and T.one().missing_cells == 0
O = next(o for o in (I.operator(record.id) for record in I.operators)
         if o.one().tensor == T.one().id and same_ids(o.rows, F0))
I.verify(O).require()
display(O.one().formula, O.related("kernel").head(), O.related("bindings").head())
assert len(O.related("kernel")) > 0
# F0 的系数变化保持为精确稀疏核；不会因共享 family 被解释为常数。
assert O.related("statistics").where(path="/min").one().number != O.related("statistics").where(path="/max").one().number
display(m.ir)


## 7. V1 是否是可消去的定义变量？

V1 的 200 个成员都以 F2×1、F5×1 的模式出现。打开一个 F2 等式，可见
一个 V1 与 950 个 V2 通过带权系数相连，因此能代数地解出该 V1。
这仍不代表代入后模型会更好；先估计新增非零项。

In [ ]:
V1 = m.family("V1@1").vars
display(V1.pattern_counts())
y = V1.iloc[0]
display(y.rows)
defining_row = y.rows.where(mf.col("row").isin(F2)).one()
print("defining row:", defining_row.name)
print(defining_row.formula(max_terms=8))
print("other uses:", (y.rows - F2).to_records())


In [ ]:
F5 = m.family("F5@2").rows
f5 = F5.one()
n_v1_in_f5 = len(f5.vars.where(var=V1))
n_v2_in_definitions = m.matrix.nnz(rows=F2, vars=V2, axis="row").column("nnz")
n_v2_already_in_f5 = len(f5.vars.where(var=V2))

print("V1 terms in F5:", n_v1_in_f5)
print("V2 terms per F2 definition:", n_v2_in_definitions.value_counts())
print("V2 terms already in F5:", n_v2_already_in_f5)
assert n_v1_in_f5 == len(V1)
assert n_v2_in_definitions.min() == n_v2_in_definitions.max() == 950
assert n_v2_already_in_f5 == 0

projected_f5_nnz = f5.nnz - n_v1_in_f5 + n_v2_in_definitions.sum()
print("F5 nnz before / after full substitution:", f5.nnz, projected_f5_nnz)
assert projected_f5_nnz == 190200


F5 原有 400 个非零项，其中 200 个是 V1。若全部 V1 代入，
200 条 F2 定义式各带来 950 个不同 V2，F5 将达到 190200 个非零项。
这个估计利用了前面验证的 F2 坐标唯一性以及 F5 原本不含 V2；
它是改写成本的结构证据，不是自动执行 substitution 的建议。

## 8. 查看 198 + 2 的边界分裂

V3 有 198 个成员：196 个关联 F3/F5，另 2 个还关联 F6。
V4 只有 2 个，均关联 F4/F5/F7。F4、F6、F7 的成员数分别为 2、1、1，
因此默认显示即可列出全部成员；长公式仍显式限项。

In [ ]:
V3 = m.family("V3@1").vars
V4 = m.family("V4@1").vars
display(V3.head())
display(V4)
display(V3.pattern_counts())
display(V4.pattern_counts())

for variable in V4:
    print(variable["name"], {key: variable[key]
                            for key in ("type", "lb", "ub", "obj", "nnz")})
    display(variable.rows)

for label in ("F4", "F6", "F7"):
    family = m.family(label + "@2").rows
    print(label)
    display(family)
    for row in family:
        print(row.name, row.formula(max_terms=8))

## 9. 最后才检查目标变量 V0

V0 的目标系数为 1，只参加 950 条 F0。F1 的 RHS 全为 1，但这不能推出
目标变量等于最大的 F1 RHS：F0 对 V2 使用变化的权重，F1 对它们使用单位系数。
必须先核对同一变量在两组约束中的真实系数，再提出 reformulation。

In [ ]:
V0 = m.family("V0@1").vars
z = V0.one()
print("objective variable:", z.name, "obj =", z.obj)
display(z.rows.value_counts("family"))

rhs = F1.column("ub")
max_rhs = rhs.max()
maximizers = rhs.argmax()
print("F1 RHS distribution:", rhs.value_counts())
print("max RHS:", max_rhs, "attained by", len(maximizers), "rows")
print("first five maximizers:", list(maximizers.head().column("name")))
assert max_rhs == 1 and len(maximizers) == len(F1)

f0_edge = x.terms.where(row=F0).one()
f1_edge = x.terms.where(row=F1).one()
print(x.name, "coef in F0:", f0_edge.coef, "coef in F1:", f1_edge.coef)
f0_row, f1_row = f0_edge.row, f1_edge.row
display(f0_row.vars.where(var=V0))
print(f0_row.name, f0_row.formula(max_terms=6))
print(f1_row.name, f1_row.formula(max_terms=6))


## 已验证的事实与下一步

- V2 与 F0×F2 构成完整的一对一坐标：190000 个变量、190000 个不同坐标。
- F0 与 F1 的 V2 support 通过全量坐标对应核验；F0 的系数存在显著数值变化。
- 200 个 V1 分别进入一条 F2 定义式和同一条 F5；全量代入会显著增加 F5 的非零项。
- V3/V4 的少量边界角色可以从邻接行直接解释；V0 的简化仍需严谨的系数推导。

下面关闭模型，释放稀疏数组、SQLite 连接和缓存。如果还要继续交互探索，
请等到最后再运行这一格。

In [ ]:
m.close()